# Session 2 — Conformational Search, QM/MM & Environment

**eChem School — Marseille 2026**

This session covers three related topics:

1. **Systematic conformational search** — use VeloxChem's `ConformerGenerator` class to exhaustively enumerate and minimize conformers.
2. **Extract conformers from an MD simulation** — Perform a MD simulation at high temperature and optimize snaphots along the trajectory to find conformers.
3. **QM/MM and environment models** — implicit solvation (CPCM, SMD) and explicit polarizable embedding (PE) with VeloxChem.

# Part 1 — Systematic Conformational Search with VeloxChem

For larger flexible molecules (e.g. drug candidates), MD is not the most efficient way to sample conformational space. A **systematic search** rotates all rotatable bonds through a grid of angles, generating all possible arrangements, and then minimizes each one.

VeloxChem provides the `ConformerGenerator` class, which:
1. Enumerates all combinations of rotatable-bond torsion angles.
2. Minimizes each conformer with an MM force field (optionally with RESP charges or an implicit solvent).
3. Filters duplicate structures by RMSD.

We demonstrate with **L-alanine** — a flexible molecule with multiple rotatable bonds.

In [ ]:
import veloxchem as vlx

# L-Alanine molecule
# TODO: Create the molecule object for L-Alanine from its SMILES string
molecule = 
molecule.show(atom_indices=True)

## 1.1 Basic conformer generation

In [ ]:
conf = vlx.ConformerGenerator()
conformers_dict = conf.generate(molecule)

# TODO: Display the number of conformers found and the global minimum conformer.


# TODO: Display the global minimum conformer using the show_global_minimum() function.


In [ ]:
# TODO: Show the three lowest-energy conformers using the show_conformers() function.


## 2.2 With RESP partial charges

By default the generator uses GAFF charges. We can improve accuracy by providing RESP charges computed at HF/6-31G*.

In [ ]:
basis = vlx.MolecularBasis.read(molecule, '6-31g*')
resp  = vlx.RespChargesDriver()
resp.ostream.mute()
# TODO: Calculate the RESP partial charges for the molecule
partial_charges = 
conf_resp = vlx.ConformerGenerator()
conf_resp.partial_charges = partial_charges
conformers_dict_resp = conf_resp.generate(molecule)

conf_resp.show_global_minimum(atom_indices=True)

## 2.3 With implicit solvent

The implicit OBC2 solvent model (a generalised Born variant) shifts conformer stabilities towards more polar geometries.

In [ ]:
conf_solv = vlx.ConformerGenerator()
conf_solv.ostream.mute()
conf_solv.show_available_implicit_solvent_models()

In [ ]:
# TODO: Set the implicit solvent model for the conformer generator
conf_solv.implicit_solvent_model = 
conformers_dict_solv = conf_solv.generate(molecule)

# TODO: Show the global minimum conformer in implicit solvent using the show_global_minimum() function.


## 2.4 Conformer extraction from high-temperature MD

For very flexible polymers, the systematic grid explodes combinatorially. A better approach is to run high-temperature MD and sample snapshots. VeloxChem's `OpenMMDynamics.conformational_sampling` does this automatically: it runs NVT MD at an elevated temperature (e.g. 1000 K), extracts snapshots, minimises each one, and optionally deduplicates identical structures.

In [ ]:
# Load a larger molecule for the MD-based conformer search
penicilin = vlx.Molecule.read_smiles('CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C')
penicilin.show()

In [ ]:
# Build force field and creating system for MD-based conformer search

ff_gen = vlx.MMForceFieldGenerator()

# TODO: Create the topology for the penicilin molecule using the force field generator


opm_dyn = vlx.OpenMMDynamics()
opm_dyn.create_system_from_molecule(penicilin,
                                    ff_gen,
                                    filename='penicilin',
                                    residue_name='MOL')

In [ ]:
# High-temperature conformational sampling
conformers_dict = opm_dyn.conformational_sampling(
    ensemble='NVT',
    temperature=1000,
    timestep=2.0,
    nsteps=10000,
    snapshots=500,
    unique_conformers=True,
    qm_driver=None,
    basis=None,
    constraints=None
)

In [ ]:
# Visualise and plot energy distribution
opm_dyn.show_conformers(number=5)

import matplotlib.pyplot as plt
import numpy as np

# TODO: Calculate the relative energies of the conformers and plot the energy distribution as a histogram.
E_rel =

plt.figure(figsize=(8, 4))
plt.hist(E_rel, bins=50, color='darkcyan', alpha=0.7)
plt.xlabel('Relative energy [kJ/mol]')
plt.ylabel('Number of conformers')
plt.title('Conformer energy distribution — penicilin')
plt.tight_layout()
plt.show()

# Save global minimum
conformers_dict['molecules'][0].show()
conformers_dict['molecules'][0].write_xyz_file('penicilin-global-min.xyz')

# Part 3 — Environment: Solvation and QM/MM

The geometry and properties of a molecule depend strongly on its environment. VeloxChem supports three levels of description:

| Level | Model | Cost |
|-------|-------|------|
| Implicit | CPCM / SMD | Cheap — continuum dielectric |
| Explicit MM | Classical point charges | Medium — no polarisation of environment |


---

## 3.1 Implicit solvation — CPCM

In the **conductor-like polarizable continuum model (CPCM)**, the solute sits inside a cavity defined by its molecular surface. The surrounding medium is treated as a conductor (dielectric constant → ∞) and the result is rescaled to the desired permittivity $\epsilon$.

VeloxChem uses CPCM for SCF energies, gradients, and linear-response spectra.

Key parameters:
- `solvation_model = "cpcm"` — activates CPCM
- `cpcm_epsilon` — solvent permittivity (default: 78.39 for water)

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name('ammonia')
basis    = vlx.MolecularBasis.read(molecule, 'def2-svp')

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = 'b3lyp'
scf_drv.solvation_model = 'cpcm'
# scf_drv.cpcm_epsilon = 24.5  # ethanol — change here for other solvents
scf_results = scf_drv.compute(molecule, basis)


## 3.2 Implicit solvation — SMD

**SMD** (Solvation Model based on Density) adds cavitation, dispersion, and short-range solute–solvent terms on top of a CPCM electrostatics calculation. It gives more accurate solvation free energies across diverse solvents.

In [ ]:
molecule_meoh = vlx.Molecule.read_name('methanol')
basis_meoh    = vlx.MolecularBasis.read(molecule_meoh, 'def2-svp')

scf_smd = vlx.ScfRestrictedDriver()
scf_smd.solvation_model = 'smd'
scf_smd.smd_solvent     = 'water'
scf_results_smd = scf_smd.compute(molecule_meoh, basis_meoh)

print('SMD energy (water):', scf_results_smd['scf_energy'], 'Hartree')

## 3.3 Explicit solvation — Non Polarizable Embedding (PE)

In the **non polarizable embedding** model, explicit solvent molecules are represented by:
- **Site charges** — reproduce the electrostatic potential

The potential file (`*.pot`) contains the environemnt. VeloxChem reads this file and incorporates the environment into the SCF and linear-response equations.

Below we prepare the potential file for a water molecule surrounded by four explicit water molecules.

In [ ]:
import py3Dmol

viewer = py3Dmol.view(width=350, height=300, linked=False)

h2o_xyz = """3
water (QM)
O        0.0000000000      0.0000000000      0.0000000000
H        0.6891400000      0.8324710000      0.0000000000
H        0.7224340000     -0.8726890000      0.0000000000
"""

solvent_xyz = """12
solvent shell (MM)
O        1.0991810000      2.2167050000      0.2791250000
H        2.2673500000      2.2529740000      0.6013580000
H        0.9984050000      2.8080510000     -0.5840690000
O       -0.8223690000      0.2141140000     -2.3767710000
H       -1.5990410000      0.8408210000     -2.6698860000
H       -0.4683410000      0.4151750000     -1.5365150000
O        1.7377440000     -2.0168380000     -0.4861090000
H        2.0753880000     -2.2446490000     -1.3654140000
H        1.6636970000     -2.7516570000      0.0543720000
O       -1.0758700000     -0.2692330000      2.5139440000
H       -0.5822670000     -0.1887880000      1.5058600000
H       -1.1632420000      0.7082890000      2.8568430000
"""

viewer.addModel(h2o_xyz,     'xyz')  # model 0: QM water
viewer.addModel(solvent_xyz, 'xyz')  # model 1: MM shell
viewer.setStyle({}, {})
viewer.addStyle({'model': 0}, {'stick': {'radius': 0.25, 'colorscheme': 'Jmol'}})
viewer.addStyle({'model': 1}, {'stick': {'radius': 0.45, 'color': '#9bbcff'}})
viewer.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
viewer.rotate(45, 'x')
viewer.zoomTo({'model': 0})
viewer.show()

In [ ]:
import veloxchem as vlx
# SCF with polarizable embedding
solute_xyz = """3
water
O        0.0000000000      0.0000000000      0.0000000000
H        0.6891400000      0.8324710000      0.0000000000
H        0.7224340000     -0.8726890000      0.0000000000
"""

# create a veloxchem *.pot file
with open("embedding.pot", "w") as pot_file:
    pot_file.write("""12
xyz
O        1.0991810000      2.2167050000      0.2791250000    -0.834
H        2.2673500000      2.2529740000      0.6013580000     0.417
H        0.9984050000      2.8080510000     -0.5840690000     0.417
O       -0.8223690000      0.2141140000     -2.3767710000    -0.834
H       -1.5990410000      0.8408210000     -2.6698860000     0.417
H       -0.4683410000      0.4151750000     -1.5365150000     0.417
O        1.7377440000     -2.0168380000     -0.4861090000    -0.834
H        2.0753880000     -2.2446490000     -1.3654140000     0.417
H        1.6636970000     -2.7516570000      0.0543720000     0.417
O       -1.0758700000     -0.2692330000      2.5139440000    -0.834
H       -0.5822670000     -0.1887880000      1.5058600000     0.417
H       -1.1632420000      0.7082890000      2.8568430000     0.417
""")

molecule = vlx.Molecule.read_xyz_string(solute_xyz)
basis = vlx.MolecularBasis.read(molecule, "cc-pvdz")
scf_drv = vlx.ScfRestrictedDriver()
scf_results = scf_drv.compute(molecule, basis)

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.point_charges = "embedding.pot"
scf_env_results = scf_drv.compute(molecule, basis)


print('Gas phase SCF energy:', scf_results['scf_energy'], 'Hartree')
print('PE SCF energy:', scf_env_results['scf_energy'], 'Hartree')

## Summary

| Topic | Key tool | What we learned |
|-------|----------|-----------------|
| Systematic conformer search | `vlx.ConformerGenerator` | Grid search + MMFF minimisation; RESP charges and implicit solvent improve results |
| MD-based conformer sampling | `vlx.OpenMMDynamics` | High-T MD with deduplication scales to large flexible molecules |
| Implicit solvation | CPCM, SMD | Cheap continuum model for SCF and spectra |
| Explicit QM/MM | Non-PE | Environment represented as PC — physically more accurate|